In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import (
    load_diabetes,
    load_linnerud,
    load_breast_cancer,
    load_wine,
    load_iris
)

np.random.seed(42)

In [ ]:
def prepare_data(X, y):
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
    X = np.c_[np.ones(len(X)), X]
    return X, y


def loss(w, X, y, problem):
    z = X @ w

    if problem == "regression":
        return np.mean((z - y) ** 2) / 2

    p = 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    return -np.mean(
        y * np.log(p + 1e-12)
        + (1 - y) * np.log(1 - p + 1e-12)
    )


def gradient(w, X, y, problem):
    z = X @ w

    if problem == "regression":
        error = z - y
    else:
        p = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
        error = p - y

    return X.T @ error / len(y)


def train(X, y, problem, method,
          learning_rate=0.01,
          epochs=50,
          batch_size=32):

    n, d = X.shape
    w = np.zeros(d)
    history = []

    if method == "Batch GD":
        bs = n
    elif method == "SGD":
        bs = 1
    else:
        bs = batch_size

    for epoch in range(epochs):

        satoru = np.random.permutation(n)

        for start in range(0, n, bs):

            idx = satoru[start:start + bs]

            X_batch = X[idx]
            y_batch = y[idx]

            g = gradient(
                w,
                X_batch,
                y_batch,
                problem
            )

            w = w - learning_rate * g

        history.append(
            loss(w, X, y, problem)
        )

    return w, history

In [ ]:
datasets = {}

X, y = load_diabetes(return_X_y=True)
y = (y - y.mean()) / y.std()

X, y = prepare_data(X, y)

datasets["Diabetes"] = (X, y, "regression")


data = load_linnerud()

X = data.data
y = data.target[:, 0].astype(float)

y = (y - y.mean()) / y.std()

X, y = prepare_data(X, y)

datasets["Linnerud"] = (X, y, "regression")


X, y = load_breast_cancer(return_X_y=True)

X, y = prepare_data(X, y.astype(float))

datasets["Breast Cancer"] = (X, y, "classification")


X, y = load_wine(return_X_y=True)

y = (y == 0).astype(float)

X, y = prepare_data(X, y)

datasets["Wine"] = (X, y, "classification")


X, y = load_iris(return_X_y=True)

y = (y == 2).astype(float)

X, y = prepare_data(X, y)

datasets["Iris"] = (X, y, "classification")

In [ ]:
methods = [
    "Batch GD",
    "SGD",
    "Mini-batch"
]

results = {}

for name, (X, y, problem) in datasets.items():

    print("\n", name)

    results[name] = {}

    for method in methods:

        w, history = train(
            X,
            y,
            problem,
            method,
            learning_rate=0.01,
            epochs=50,
            batch_size=32
        )

        results[name][method] = history

        print(
            method,
            "Final Loss =",
            history[-1]
        )

In [ ]:
fig, axes = plt.subplots(
    1, 5,
    figsize=(22, 4)
)

for ax, (name, data) in zip(
        axes, results.items()):

    for method in methods:
        ax.plot(
            data[method],
            label=method
        )

    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()